# Manim Workshop

From one circle to Fourier epicycles tracing a photograph.

This notebook is **generated** from `scenes/` by `scripts/build_notebook.py` —
the code in every cell below is the same code the repository ships, so you can
copy any cell into a `.py` file and run it with the `manim` CLI instead.

## Before you start

```bash
uv venv --python 3.13 && source .venv/bin/activate
uv pip install -r requirements.txt && uv pip install -e .
pip install -r requirements-dev.txt        # JupyterLab + ipykernel
python -m ipykernel install --user --name manim-workshop --display-name "Python (manim-workshop)"
```

Then choose **Kernel → Change Kernel → Python (manim-workshop)**. The
`pip install -e .` line is what makes `manim_workshop` importable in the last
section. Full details: `docs/01-getting-started.md`.


In [ ]:
from manim import *
import numpy as np

config.max_files_cached = 200


## Make a circle

A **mobject** is any mathematical object. `Create` draws it, and `rate_func` controls *how* it is drawn.

In [ ]:
%%manim -ql HelloCircle
class HelloCircle(Scene):
    """Draw a circle — slowly, with a deliberately pleasing rate function.

    Concepts: ``Circle``, ``Create``, ``run_time``, ``rate_func``.
    """

    def construct(self):
        circle = Circle(radius=2)
        self.play(
            Create(circle),
            run_time=2,
            rate_func=rate_functions.ease_in_out_circ,
        )

## Fill it

`.animate` interpolates any property you can set. A fill needs an `opacity`.

In [ ]:
%%manim -ql FilledCircle
class FilledCircle(Scene):
    """Recolour an existing circle, then give it a translucent fill.

    Concepts: ``.animate``, ``set_color``, ``set_fill`` (note that a fill needs
    an explicit ``opacity`` or it stays invisible).

    ``self.add(circle)`` would drop the circle on screen instantly instead of
    animating it in — swap the first ``play`` for an ``add`` to see the
    difference.
    """

    def construct(self):
        circle = Circle(radius=2)
        self.play(
            Create(circle),
            run_time=2,
            rate_func=rate_functions.ease_in_out_circ,
        )
        self.play(circle.animate.set_color(GREEN).set_fill(ORANGE, opacity=0.5))

## A square becomes a circle

`Transform(a, b)` morphs `a` into `b` and leaves `a` on screen — keep animating `a` afterwards.

In [ ]:
%%manim -ql SquareToCircle
class SquareToCircle(Scene):
    """Rotate a square, colour it, then ``Transform`` it into a circle.

    Concepts: ``Square``, ``rotate``, ``Transform`` — and the classic gotcha that
    after ``Transform(square, circle)`` you must animate *``square``* (the
    on-screen mobject), not ``circle`` (the template).
    """

    def construct(self):
        circle = Circle(radius=2)
        square = Square()
        square.rotate(PI / 6)

        self.play(Create(square))
        self.play(square.animate.set_color(GREEN))
        self.play(Transform(square, circle))
        self.play(square.animate.set_fill(GREEN, opacity=0.5))

## Put two shapes together

`next_to` positions one mobject relative to another; `buff=0` makes them touch.

In [ ]:
%%manim -ql CircleAndSquare
class CircleAndSquare(Scene):
    """Place a square directly below a circle, then fill the circle.

    Concepts: ``next_to(mob, direction, buff=...)``, ``DOWN``, draw order
    (``Create`` order decides what is on top).
    """

    def construct(self):
        circle = Circle(radius=1)
        square = Square()
        square.rotate(PI / 4)
        square.next_to(circle, DOWN, buff=0)

        self.play(Create(square))
        self.play(Create(circle))
        self.play(circle.animate.set_fill(GREEN, opacity=0.5))

## Groups and rotation

A `VGroup` animates several mobjects as one. `about_point` chooses the pivot.

In [ ]:
%%manim -ql CircleAndSquareRotating
class CircleAndSquareRotating(Scene):
    """Group a circle and a square, then spin the group around the origin.

    Concepts: ``VGroup``, ``Rotate(..., about_point=ORIGIN, axis=OUT)``.
    """

    def construct(self):
        circle = Circle(radius=1)
        square = Square()
        square.rotate(PI / 4)
        square.next_to(circle, DOWN, buff=0)

        group = VGroup(circle, square)

        self.play(Create(group), run_time=2)
        self.play(circle.animate.set_fill(GREEN, opacity=0.5))
        self.play(Rotate(group, angle=2 * PI, about_point=ORIGIN, axis=OUT, run_time=5))

## Leave a trail

`TracedPath` records where a point has been. Add it first so it draws behind.

In [ ]:
%%manim -ql CircleAndSquareRotatingWithTracing
class CircleAndSquareRotatingWithTracing(Scene):
    """The same spin, but the square's centre leaves a trail behind it.

    Concepts: ``TracedPath(mob.get_center, stroke_color=..., stroke_width=...)``
    — add the path *before* the motion so it is behind everything else.
    """

    def construct(self):
        circle = Circle(radius=1)
        square = Square()
        square.rotate(PI / 4)
        square.next_to(circle, DOWN, buff=0)

        group = VGroup(circle, square)

        path = TracedPath(square.get_center, stroke_color=BLUE, stroke_width=2)
        self.add(path)

        self.play(Create(group), run_time=2)
        self.play(circle.animate.set_fill(GREEN, opacity=0.5))
        self.play(Rotate(group, angle=2 * PI, about_point=ORIGIN, axis=OUT, run_time=5))

        self.play(FadeOut(square))

## An orbit

The pivot is one of the circles, so the pair revolves instead of spinning in place.

In [ ]:
%%manim -ql CircleAndCircleRotating
class CircleAndCircleRotating(Scene):
    """Two touching circles: rotate the pair around the *right* circle's centre.

    The pair is shifted left first, which makes the anchor obvious on screen —
    ``about_point`` is what turns "a rotation" into "an orbit".
    """

    def construct(self):
        c1 = Circle(radius=0.5)
        c1.set_fill(GREEN, opacity=0.6)
        c2 = Circle(radius=0.7)
        c2.set_fill(YELLOW, opacity=0.6)
        c1.next_to(c2, LEFT)

        group = VGroup(c1, c2)
        group.shift(5 * LEFT)

        self.play(Create(group), run_time=2)
        self.play(
            Rotate(
                group, angle=2 * PI, about_point=c2.get_center(), axis=OUT, run_time=5
            )
        )

        self.play(FadeOut(c2))
        self.play(FadeOut(c1))

## Moving

`.animate.move_to` is absolute; `shift` is relative.

In [ ]:
%%manim -ql CircleAndCircleMoving
class CircleAndCircleMoving(Scene):
    """The same pair, translated instead of rotated: ``.animate.move_to``.

    Concepts: ``mob.animate.move_to(point)`` — the group keeps its internal
    layout while its centre travels. Compare with ``shift``, which is relative.
    """

    def construct(self):
        c1 = Circle(radius=0.5)
        c1.set_fill(GREEN, opacity=0.6)
        c2 = Circle(radius=0.7)
        c2.set_fill(YELLOW, opacity=0.6)
        c1.next_to(c2, LEFT)

        group = VGroup(c1, c2)
        group.shift(5 * LEFT)

        self.play(Create(group), run_time=2)
        self.play(group.animate.move_to(5 * RIGHT))

        self.play(FadeOut(c2))
        self.play(FadeOut(c1))

## Spin *and* move

`add_updater` runs code every frame, so rotation and translation can overlap — and the trace compounds into a cycloid.

In [ ]:
%%manim -ql CircleAndDotRotatingAndTracing
class CircleAndDotRotatingAndTracing(Scene):
    """An ``add_updater`` spin that never stops, while the group travels right.

    Concepts: ``mob.add_updater(lambda mob, dt: ...)`` runs on *every* frame
    (``dt`` is the time since the last frame), so rotation and translation can
    overlap — and ``TracedPath`` records the compounded path.
    """

    def construct(self):
        circle = Circle(radius=0.5).shift(LEFT * 5)
        dot = Dot(circle.get_left())
        group = VGroup(circle, dot)

        trace = TracedPath(dot.get_center)

        group.add_updater(lambda mob, dt: mob.rotate(3 * dt))

        self.add(trace)
        self.play(FadeIn(group))
        self.play(group.animate.shift(RIGHT * 10), run_time=7)

## Projects: the sine function

A `ValueTracker` is an abstract time axis; `always_redraw` rebuilds the dot from it each frame.

In [ ]:
%%manim -ql SineSnake
class SineSnake(Scene):
    """One period of a sine wave, drawn by a travelling dot.

    Concepts: ``ValueTracker`` as an abstract time variable, ``always_redraw``,
    ``TracedPath(dissipating_time=...)`` for a fading comet tail.

    Try: change ``run_time`` on the final ``play``; the curve is the same, the
    *pacing* is not.
    """

    def construct(self):
        tracker = ValueTracker(-2 * PI)

        dot = always_redraw(
            lambda: Dot(
                [tracker.get_value(), np.sin(tracker.get_value()), 0],
                color=YELLOW,
            )
        )
        tracer = TracedPath(
            dot.get_center, stroke_color=RED, stroke_width=9, dissipating_time=1.5
        )

        self.add(dot)
        self.add(tracer)
        self.play(tracker.animate.set_value(2 * PI), run_time=15)

## Epicycloid

Two rotations nest here: the small wheel orbits, the pen spins.

In [ ]:
%%manim -ql Epicycloid
class Epicycloid(Scene):
    """A circle orbiting a circle: the classic "spirograph" curve.

    Two rotations nest here — one updater spins the small circle around the big
    one (``f1`` turns), a second spins the dot around the small circle (``f2``
    turns). Their ratio is the whole design space.

    Concepts: two independent ``add_updater`` rotation rates, ``clear_updaters``.
    """

    def construct(self):
        r1, r2 = 0.9, 0.6  # radii of the big wheel and the small wheel
        f1, f2 = 3, 8  # turns per second: orbit rate, spin rate

        c1 = Circle(radius=r1)
        c2 = Circle(radius=r2).next_to(c1, LEFT, buff=0)
        dot = Dot(c2.get_left())
        trace = TracedPath(dot.get_center, dissipating_time=4)

        self.play(FadeIn(dot), run_time=0.25)
        self.play(FadeIn(c2), run_time=0.25)
        self.play(FadeIn(c1), run_time=0.25)

        wheel = VGroup(c2, dot)
        wheel.add_updater(
            lambda mob, dt: mob.rotate(f1 * dt, about_point=c1.get_center())
        )
        dot.add_updater(
            lambda mob, dt: mob.rotate(f2 * dt, about_point=c2.get_center())
        )

        self.add(wheel)
        self.add(trace)

        self.wait(15)

        # Updaters keep running until you stop them.
        wheel.clear_updaters()
        dot.clear_updaters()
        self.play(FadeOut(wheel), FadeOut(trace), run_time=0.5)
        self.play(FadeOut(dot), run_time=0.5)

## Lissajous figures

Two sines, one per axis. Rational ratios close the curve; irrational ones never do.

In [ ]:
%%manim -ql LissajousFigures
class LissajousFigures(Scene):
    """Two sine waves, one per axis, out of phase — a Lissajous figure.

    Concepts: independent functions of the *same* tracker parameter; ``phi``
    shifts one axis in time. Changing ``w1``/``w2`` changes the knot; a rational
    ratio closes the curve.

    Try: ``w1=3, w2=4`` (this one), ``w1=3, w2=5``, ``phi=0``.
    """

    def construct(self):
        w1, w2 = 3, 4
        phi = PI / 3

        tracker = ValueTracker(-2 * PI)

        dot = always_redraw(
            lambda: Dot(
                [
                    np.sin(w1 * tracker.get_value()),
                    np.sin(w2 * tracker.get_value() + phi),
                    0,
                ],
                color=YELLOW,
            )
        )
        tracer = TracedPath(
            dot.get_center, stroke_color=RED, stroke_width=9, dissipating_time=None
        )

        self.add(dot)
        self.add(tracer)
        self.play(tracker.animate.set_value(2 * PI), run_time=20)

## A rose

The polar equation `r = sin(k·θ)`, written out in Cartesian coordinates.

In [ ]:
%%manim -ql RoseCurve
class RoseCurve(Scene):
    """A rose: polar ``r = sin(k·θ)`` walked in Cartesian coordinates.

    ``k`` is the petal count (for odd ``k``). The two coordinates are the polar
    recipe ``(r·cos θ, r·sin θ)`` written out explicitly.

    Concepts: parametric plotting from polar equations.
    """

    def construct(self):
        k = 5
        scale = 3

        tracker = ValueTracker(-2 * PI)

        dot = always_redraw(
            lambda: Dot(
                [
                    scale
                    * np.sin(k * tracker.get_value())
                    * np.cos(tracker.get_value()),
                    scale
                    * np.sin(k * tracker.get_value())
                    * np.sin(tracker.get_value()),
                    0,
                ],
                color=YELLOW,
            )
        )
        tracer = TracedPath(
            dot.get_center, stroke_color=RED, stroke_width=9, dissipating_time=None
        )

        self.add(dot)
        self.add(tracer)
        self.play(tracker.animate.set_value(2 * PI), run_time=20)

## Flowers: colour per petal

Each petal gets its own `TracedPath`, so each stroke has its own colour. `time_period` is what makes the petals meet cleanly.

In [ ]:
%%manim -ql FlowerWithColors
class FlowerWithColors(Scene):
    """A five-petal rose, coloured petal by petal.

    The loop walks the tracker forward one petal-period at a time; because each
    petal gets its own ``TracedPath`` with its own colour, the same dot draws a
    multi-coloured flower.

    Concepts: segments, ``for`` loops, ``time_period`` for an odd petal count.
    """

    def construct(self):
        colors = [PINK, RED, ORANGE, GOLD, YELLOW]
        k = 5
        scale = 3

        # Odd k closes after PI, even k needs the full 2*PI.
        time_period = PI / k if k % 2 == 1 else 2 * PI / k
        tracker = ValueTracker(0)

        for x in range(k):
            petal_color = colors[x % len(colors)]
            dot = always_redraw(
                lambda: Dot(
                    [
                        scale
                        * np.sin(k * tracker.get_value())
                        * np.cos(tracker.get_value()),
                        scale
                        * np.sin(k * tracker.get_value())
                        * np.sin(tracker.get_value()),
                        0,
                    ]
                )
            )
            path = TracedPath(dot.get_center, stroke_width=9, stroke_color=petal_color)
            self.add(dot, path)

            self.play(tracker.animate.set_value((x + 1) * time_period))

## Rotating flower

The traced petals are copied into one `VMobject` with no updaters attached, then rotated.

In [ ]:
%%manim -ql RotatingFlower
class RotatingFlower(Scene):
    """A twelve-petal rose, then the scaffolding is replaced by one object.

    After the petals are traced, every path is copied into a plain ``VMobject``
    (points only, no updaters), the dot/path originals are removed, and the
    assembled flower is rotated a quarter turn as a rigid body.

    Concepts: ``VGroup`` as a container, ``set_points``/``get_points`` to clone
    geometry, ``remove`` to drop the machinery.
    """

    def construct(self):
        colors = [PINK, RED, ORANGE, GOLD, YELLOW]
        k = 12
        scale = 3

        time_period = PI / k if k % 2 == 1 else 2 * PI / k
        tracker = ValueTracker(0)

        segments = VGroup()
        for x in range(k):
            petal_color = colors[x % len(colors)]
            dot = always_redraw(
                lambda: Dot(
                    [
                        scale
                        * np.sin(k * tracker.get_value())
                        * np.cos(tracker.get_value()),
                        scale
                        * np.sin(k * tracker.get_value())
                        * np.sin(tracker.get_value()),
                        0,
                    ]
                )
            )
            path = TracedPath(dot.get_center, stroke_width=9, stroke_color=petal_color)
            self.add(dot, path)
            segments.add(dot, path)

            self.play(tracker.animate.set_value((x + 1) * time_period))

        flower = VGroup(
            *[
                VMobject(
                    stroke_color=segment.stroke_color,
                    stroke_width=segment.stroke_width,
                ).set_points(segment.get_points())
                for segment in segments
            ]
        )
        self.remove(tracker, *segments)
        self.add(flower)

        self.play(Rotate(flower, angle=PI / 2, about_point=ORIGIN, run_time=4))

## Gradient flower

`stroke_color` accepts a *list* of colours and interpolates along the stroke.

In [ ]:
%%manim -ql FlowerWithGradient
class FlowerWithGradient(Scene):
    """The twelve-petal rose again — this time every stroke is a gradient.

    ``stroke_color`` accepts a *list* of colours, and Manim interpolates along
    the stroke. Handing petal ``x`` the pair ``(colors[x], colors[x+1])`` makes
    the whole flower cycle smoothly through the palette.

    Concepts: gradient strokes, glow via ``stroke_width``.
    """

    def construct(self):
        colors = [PINK, RED, ORANGE, GOLD, YELLOW]
        k = 12
        scale = 3

        time_period = PI / k if k % 2 == 1 else 2 * PI / k
        tracker = ValueTracker(0)

        segments = VGroup()
        for x in range(k):
            dot = always_redraw(
                lambda: Dot(
                    [
                        scale
                        * np.sin(k * tracker.get_value())
                        * np.cos(tracker.get_value()),
                        scale
                        * np.sin(k * tracker.get_value())
                        * np.sin(tracker.get_value()),
                        0,
                    ]
                )
            )
            path = TracedPath(
                dot.get_center,
                stroke_width=9,
                stroke_color=[colors[x % len(colors)], colors[(x + 1) % len(colors)]],
            )
            self.add(dot, path)
            segments.add(dot, path)

            self.play(tracker.animate.set_value((x + 1) * time_period))

        flower = VGroup(
            *[
                VMobject(
                    stroke_color=segment.stroke_color,
                    stroke_width=segment.stroke_width,
                ).set_points(segment.get_points())
                for segment in segments
            ]
        )
        self.remove(tracker, *segments)
        self.add(flower)

        self.play(Rotate(flower, angle=PI / 2, about_point=ORIGIN, run_time=4))

## The finale: Fourier epicycles

Everything so far has been shapes. This last one is the trick from 3Blue1Brown's
*"But what is a Fourier series?"*: a chain of rotating circles, whose radii and
phases come from a Fourier transform, draws any closed outline you give it.

The maths now lives in the package rather than in this notebook (one
implementation, used by both the notebook and `scenes/07_fourier_epicycle.py`).
The annotated walk-through is in `docs/concepts/07-fourier-epicycles.md`.


In [ ]:
from manim_workshop.fourier import get_fourier_coefficients

# The bundled silhouette: a dark subject on a light background.
coeffs = get_fourier_coefficients()
print(f"{len(coeffs)} circles; the biggest has radius {coeffs[0]['radius']:.3f}")


In [ ]:
%%manim -ql FourierEpicycle
class FourierEpicycle(Scene):
    """Draw the Doraemon outline with ~61 rotating epicycles.

    Concepts: DFT of a sampled contour, tip-to-tail vector addition,
    ``always_redraw`` for a cheap "recompute everything each frame" scene.

    Try: ``get_fourier_coefficients(num_coeffs=200)`` for a sharper outline (and
    a slower render), or point it at your own image in ``assets/``.
    """

    def construct(self):
        coeffs = get_fourier_coefficients()
        tracker = ValueTracker(0)

        def get_outline():
            """One frame: the epicycle chain at the tracker's current time."""
            current_center = np.array([0, 0, 0])
            group = VGroup()

            for coeff in coeffs:
                t = tracker.get_value()
                radius = coeff["radius"]
                frequency = coeff["freq"]
                phase = coeff["phase"]

                angle = frequency * t + phase
                next_center = (
                    np.array([radius * np.cos(angle), radius * np.sin(angle), 0])
                    + current_center
                )

                circle = Circle(
                    radius=radius,
                    color=BLUE,
                    stroke_opacity=0.4,
                    stroke_width=0.4,
                ).move_to(current_center)
                spoke = Line(
                    current_center,
                    next_center,
                    color=WHITE,
                    stroke_opacity=0.6,
                    stroke_width=0.4,
                )
                group.add(circle, spoke)
                current_center = next_center

            tip = Dot(color=YELLOW, point=current_center)
            group.add(tip)
            return group

        outline = always_redraw(get_outline)

        # The pen sits on the last link's free end; the trail is what we keep.
        dot = Dot().add_updater(lambda mob: mob.move_to(outline[-1].get_center()))
        trace = TracedPath(dot.get_center, stroke_width=9, stroke_color=GOLD)

        self.add(outline, dot, trace)
        self.play(tracker.animate.set_value(2 * PI), run_time=20)

## Where next

* `docs/03-scene-catalog.md` — every scene, with the API it teaches and a run command.
* `docs/concepts/` — seven notes, including the DFT derivation behind the finale.
* Exercises are at the bottom of each concept note and in
  `docs/02-workshop-outline.md`.

Point the finale at your own drawing: put a dark shape on a light background into
`assets/`, then call `get_fourier_coefficients("my_drawing.png")`.
